In [1]:
#Rag Pipeline
from loaders import UniversalLoader
from embedders import OllamaEmbedder
from vectorstores import ChromaVectorStore
from ingesters import GlobIngester
import chunkers
import agents

from openai import OpenAI
import gradio as gr

import os
from dotenv import load_dotenv
    

C:\Users\Seth\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Set up environment variables

In [2]:
load_dotenv(override=True)
openai_api_key = os.getenv("OPENAI_API_KEY")


### Initialization of Pipeline Class Instances for Ingestion

In [ ]:
#Create Instances of Tools
embedder = OllamaEmbedder(model_name="nomic-embed-text")
vectorstore = ChromaVectorStore.ephemeral(embedder, collection_name="TemplateDB")
loader = UniversalLoader()
chunker = chunkers.FixedSizeChunkStrategy(max_tokens=800, overlap_tokens=150)
ingester = GlobIngester(roots=["./TestDocuments"], doctype_filter={"pdf", "docx", "txt"})

### Document Ingestion

In [ ]:
#Find files and load into database
files = ingester.collect()

for file in files:
    document = loader.load(file)
    chunks = chunker.chunk(document)
    result = vectorstore.upsert(chunks)
    print(f"{file.name}: {result}")

In [5]:
#Test Vector Store
vectorstore.query(query = "What's my dogs name?", top_k=1)['documents'][0]

['I am 29 years old\nI have dog named Ralph\nMy eyes are green\nI was born in May\nMy favorite color is forest green\nI am an engineer\nI have no hobbies\nMy name is sharko']

### Create Tools

In [ ]:
information_lookup_desc = "Search the vector database for information if you do not have sufficient information to answer the question."
information_lookup_params = {
    "properties": {
        "query": {"type": "string", "description": "Natural language query to search."},
        "top_k": {"type": "integer", "description": "Number of results to return.", "default": 5, "maximum": 50}
    },
    "required": ["query"]
}

# function wrapper
information_lookup_function = lambda query, top_k=1: str(vectorstore.query(query, top_k)['documents'][0])

# Creation of tool object
information_lookup = agents.Tool(
    function=information_lookup_function,
    name="Information_Lookup",
    description=information_lookup_desc,
    parameters=information_lookup_params
)

### Create Agents

In [7]:
if openai_api_key :
  print("Using OpenAI API Client")
  openai_base_url = "https://api.openai.com/v1"
  client = OpenAI(api_key=openai_api_key, base_url= openai_base_url)
  model = "gpt-4o-mini"
else:
  print("Using Ollama Local LLM Client")
  ollama_url = "http://localhost:11434/v1"
  client = OpenAI(api_key="ollama",base_url=ollama_url)
  model = "ministral-3:8b"

chatbot_prompt = """
   You are a helpful AI with access to user information.  Use the information lookup tool to answer personal questions related to the user"
 """
#create instance of chat agent
chatbot = agents.ChatAgent(system_prompt = chatbot_prompt, model=model, client=client, tools=information_lookup)

#single call to the agent
response = chatbot.call("howdy!, what is my favorite color", stream=False)

print(response)

Using OpenAI API Client


BadRequestError: Error code: 400 - {'error': {'message': "Missing parameter 'tool_call_id': messages with role 'tool' must have a 'tool_call_id'.", 'type': 'invalid_request_error', 'param': 'messages.[2].tool_call_id', 'code': None}}

### Gradio UI

In [8]:
import logging
logging.basicConfig(level="INFO")
def chatwrapper(message, history):
    print(message)
    response = []
    for token in chatbot.chat(message, stream=True):
        if token:
            response.append(token)
            yield "".join(response)

demo = gr.ChatInterface(fn=chatwrapper, chatbot=gr.Chatbot()).launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


Howdy partener
what is my name


Traceback (most recent call last):
  File "C:\Users\Seth\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\gradio\queueing.py", line 766, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\Seth\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\gradio\route_utils.py", line 355, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\Seth\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\gradio\blocks.py", line 2152, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\Seth\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\Local